<a href="https://colab.research.google.com/github/Eva012400/annotation-data-SDL/blob/main/Qwen%20Embedding%200.6B%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
!pip install -U "transformers>=4.51.0" "sentence-transformers>=2.7.0" accelerate

In [1]:
!nvidia-smi

Mon Apr 27 18:19:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
df = pd.read_parquet("party_text.parquet")

df.head(10)

,year,respondent_party,like_pty_dem_text,like_pty_rep_text,dislike_pty_dem_text,dislike_pty_rep_text,democrat_no_meaningful_answer,democrat_can_answer,rep_no_meaningful_answer,rep_can_answer
0,1984,Republican,,Budget minded. A little more conscious of othe...,Spend too much money.,,False,True,False,True
1,1984,,The Democratic party is a compassionate party....,The Republican party encourages private compet...,The great giveaway program It's not like they ...,,False,True,False,True
2,1984,Democrat,They're striving to help individuals They stre...,,,,False,True,True,True
3,1984,Democrat,Democrats are mostly for helping and providing...,,,,False,True,True,True
4,1984,Republican,,They are more fiscally responsible. They are n...,Reputation for spending and not showing where ...,The president has not used very good judgement...,False,True,False,True
5,1984,Democrat,,,,,True,True,True,True
6,1984,Democrat,He gets up there and tells you if he's doing t...,They have tried to help in some ways. They hav...,They all spend too much money.,He thinks about overseas more than at home. He...,False,True,False,True
7,1984,,,,,,True,True,True,True
8,1984,Republican,They encompass a larger array of ethnic groups...,They are identified with successful people and...,I disagree with many of their policies. Their ...,"Although like to identify with the elite, I di...",False,True,False,True
9,1984,Republican,,,,,True,True,True,True


In [4]:
# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


Using device: cuda
GPU: Tesla T4


In [5]:
text_cols = [
    "like_pty_dem_text",
    "like_pty_rep_text",
    "dislike_pty_dem_text",
    "dislike_pty_rep_text"
]

question_labels = {
    "like_pty_dem_text": "like_dem",
    "like_pty_rep_text": "like_rep",
    "dislike_pty_dem_text": "dislike_dem",
    "dislike_pty_rep_text": "dislike_rep"
}

df_long = df.melt(
    id_vars=["year", "respondent_party"],
    value_vars=text_cols,
    var_name="question",
    value_name="text"
)

df_long["question"] = df_long["question"].map(question_labels)
df_long["text"] = df_long["text"].astype("string")
df_long["respondent_party"] = df_long["respondent_party"].astype("string")

df_clean = df_long[
    df_long["text"].notna() &
    (df_long["text"].str.strip() != "") &
    (df_long["text"].str.strip().str.upper() != "SK") &
    (df_long["text"].str.strip().str.upper() != "NA") &
    df_long["respondent_party"].notna() &
    (df_long["respondent_party"].str.strip() != "")
].copy()

model = SentenceTransformer("all-MiniLM-L6-v2")

df_clean["embedding"] = list(
    model.encode(
        df_clean["text"].tolist(),
        show_progress_bar=True,
        convert_to_numpy=True
    )
)

all_rows = []
summary_rows = []

for (year, party, question), subdf in df_clean.groupby(
    ["year", "respondent_party", "question"]
):
    emb_matrix = np.vstack(subdf["embedding"].values)
    center = emb_matrix.mean(axis=0).reshape(1, -1)

    sims = cosine_similarity(emb_matrix, center).flatten()
    dists = 1 - sims

    temp = subdf.copy()
    temp["cosine_distance_to_group_center"] = dists
    all_rows.append(temp)

    summary_rows.append({
        "year": year,
        "respondent_party": party,
        "question": question,
        "n_texts": len(subdf),
        "avg_cosine_distance": dists.mean(),
    })

df_with_distances = pd.concat(all_rows, ignore_index=True)
summary_df = pd.DataFrame(summary_rows)

print(summary_df.head())
print(df_with_distances.head())


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

KeyboardInterrupt: 

In [5]:
text_cols = [
    "like_pty_dem_text",
    "like_pty_rep_text",
    "dislike_pty_dem_text",
    "dislike_pty_rep_text"
]

question_labels = {
    "like_pty_dem_text": "like_dem",
    "like_pty_rep_text": "like_rep",
    "dislike_pty_dem_text": "dislike_dem",
    "dislike_pty_rep_text": "dislike_rep"
}

df_long = df.melt(
    id_vars=["year", "respondent_party"],
    value_vars=text_cols,
    var_name="question",
    value_name="text"
)

df_long["question"] = df_long["question"].map(question_labels)
df_long["text"] = df_long["text"].astype("string")
df_long["respondent_party"] = df_long["respondent_party"].astype("string")

df_clean = df_long[
    df_long["text"].notna() &
    (df_long["text"].str.strip() != "") &
    (df_long["text"].str.strip().str.upper() != "SK") &
    (df_long["text"].str.strip().str.upper() != "NA") &
    df_long["respondent_party"].notna() &
    (df_long["respondent_party"].str.strip() != "")
].copy()

# Load Qwen3-Embedding-8B on GPU
model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B",
    device=device,
    model_kwargs={"torch_dtype": torch.float16}
)

print("Model loaded on:", model.device)

# Encode texts with Qwen Embedding 8B
texts = df_clean["text"].astype(str).tolist()

embeddings = model.encode(
    texts,
    batch_size=1,                 # important for T4 GPU
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

df_clean["embedding"] = list(embeddings)

all_rows = []
summary_rows = []

for (year, party, question), subdf in df_clean.groupby(
    ["year", "respondent_party", "question"]
):
    emb_matrix = np.vstack(subdf["embedding"].values)

    center = emb_matrix.mean(axis=0).reshape(1, -1)

    sims = cosine_similarity(emb_matrix, center).flatten()
    dists = 1 - sims

    temp = subdf.copy()
    temp["cosine_distance_to_group_center"] = dists
    all_rows.append(temp)

    summary_rows.append({
        "year": year,
        "respondent_party": party,
        "question": question,
        "n_texts": len(subdf),
        "avg_cosine_distance": dists.mean(),
    })

df_with_distances = pd.concat(all_rows, ignore_index=True)
summary_df = pd.DataFrame(summary_rows)

print(summary_df.head())
print(df_with_distances.head())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Model loaded on: cuda:0


Batches:   0%|          | 0/69945 [00:00<?, ?it/s]

   year respondent_party     question  n_texts  avg_cosine_distance
0  1984         Democrat  dislike_dem      282             0.356135
1  1984         Democrat  dislike_rep      507             0.328506
2  1984         Democrat     like_dem      608             0.318401
3  1984         Democrat     like_rep      216             0.352907
4  1984       Republican  dislike_dem      433             0.349936
   year respondent_party     question  \
0  1984         Democrat  dislike_dem   
1  1984         Democrat  dislike_dem   
2  1984         Democrat  dislike_dem   
3  1984         Democrat  dislike_dem   
4  1984         Democrat  dislike_dem   

                                                text  \
0                     They all spend too much money.   
1  The way they are handling the presidential cam...   
2  They are too liberal. The welfare state got ou...   
3  When things get better and the economy is back...   
4  Sometimes they are a little too liberal. They ...   

        

In [6]:
# df_with_distances.to_parquet("texts_with_cosine_distance.parquet", index=False)
summary_df.to_csv("avg_cosine_distance_by_year_party-3.csv", index=False)
display(summary_df)

,year,respondent_party,question,n_texts,avg_cosine_distance
0,1984,Democrat,dislike_dem,282,0.356135
1,1984,Democrat,dislike_rep,507,0.328506
2,1984,Democrat,like_dem,608,0.318401
3,1984,Democrat,like_rep,216,0.352907
4,1984,Republican,dislike_dem,433,0.349936
...,...,...,...,...,...
99,2024,Democrat,like_rep,663,0.339854
100,2024,Republican,dislike_dem,1730,0.330573
101,2024,Republican,dislike_rep,1095,0.331661
102,2024,Republican,like_dem,592,0.335771
